# Laboratory 08 — Entropy and multiplicity

In this laboratory you will count microstates directly, watch the Boltzmann entropy emerge
from that counting, and use the same counting to break two common intuitions about entropy.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. That is not a ritual: a prediction you
have committed to is the only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | $N$ distinguishable objects, each in one of two states |
| **Dynamics** | for the sampled version, one randomly chosen object switches state per step |
| **Boundary** | closed; $N$ is fixed |
| **Ensemble** | microcanonical in the sense that every microstate is equally probable — the fundamental assumption this module examines |
| **Ignored** | interactions between objects, energy differences between the two states |
| **Valid when** | the two states are energetically equivalent and objects are independent |
| **Failure modes** | interacting systems, or states with different energies, where the Boltzmann factor takes over |

All the physics lives in `thermolab.multiplicity` — open it and read it. Nothing in this course
is hidden inside a framework.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import gammaln

from thermolab import multiplicity
from thermolab.constants import K_B
from thermolab.validation import relative_error, scaling_exponent

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"k_B = {K_B:.6e} J/K")

## Part 1 — Counting microstates by hand

Start small enough to count directly. Ten distinguishable objects, each independently in one
of two states, gives $2^{10} = 1024$ microstates in total.

In [ ]:
N_SMALL = 10
counts = np.arange(N_SMALL + 1)
omegas = np.array([multiplicity.multiplicity(N_SMALL, int(n)) for n in counts])

for n, omega in zip(counts, omegas):
    print(f"n = {n:2d}   Omega(10, n) = {omega:6.0f}   P(n) = {omega / 2**N_SMALL:.5f}")

print(f"\nsum of Omega(10, n) = {omegas.sum():.0f}   (2^10 = {2**N_SMALL})")

plt.figure(figsize=(5.5, 3.5))
plt.bar(counts, omegas, color="#2563eb")
plt.xlabel("n (objects in state 1)")
plt.ylabel(r"$\Omega(10, n)$")
plt.title("Multiplicity of every macrostate, N = 10")
plt.tight_layout()
plt.show()

The macrostate $n=5$ alone accounts for about a quarter of all $1024$ microstates, while the
fully ordered extremes $n=0$ and $n=10$ each account for about one in a thousand. This gap is
the entire content of the module, already visible at $N=10$.

### Predict

Before running the next cell, write down what you expect to happen to the *relative* width of
this peak, $\sigma/(N/2)$, as $N$ grows from $100$ to $10{,}000$ to $1{,}000{,}000$. Will the
peak's location move? Will its relative width shrink, grow, or stay the same?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
sizes = np.array([100, 1_000, 10_000, 100_000, 1_000_000])
widths = np.array([multiplicity.peak_relative_width(int(n)) for n in sizes])
exponent = scaling_exponent(sizes, widths)

for n, w in zip(sizes, widths):
    print(f"N = {n:>9}   sigma/(N/2) = {w:.6f}   1/sqrt(N) = {1 / np.sqrt(n):.6f}")
print(f"\nfitted exponent = {exponent:.6f}   (theory: -0.5)")

plt.figure(figsize=(5.5, 4))
plt.loglog(sizes, widths, "o", label="peak_relative_width(N)")
plt.loglog(sizes, 1 / np.sqrt(sizes), "-", label=r"$N^{-1/2}$")
plt.xlabel("N")
plt.ylabel(r"$\sigma / (N/2)$")
plt.title(f"relative peak width (fitted slope {exponent:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

The peak never moves — it always sits at $n=N/2$. What changes is how sharply everything else
is excluded, and it falls off as $N^{-1/2}$: the same law that governed pressure fluctuations
in module 04, here derived from nothing but combinatorics.

## Part 2 — Boltzmann entropy

$S = k_B \ln \Omega$. Compute it across every macrostate of a fixed-size system and watch it
peak, symmetrically, at the even split, vanishing exactly at the fully ordered extremes.

In [ ]:
N_ENTROPY = 200
n_values = np.arange(N_ENTROPY + 1)
entropies_over_kb = np.array([multiplicity.log_multiplicity(N_ENTROPY, int(n)) for n in n_values])

plt.figure(figsize=(6, 4))
plt.plot(n_values, entropies_over_kb, color="#2563eb")
plt.axvline(N_ENTROPY / 2, color="crimson", ls="--", lw=1.0)
plt.xlabel("n")
plt.ylabel(r"$S(N, n) / k_B$")
plt.title(f"Entropy across every macrostate, N = {N_ENTROPY}")
plt.tight_layout()
plt.show()

print(f"S(N, 0)      = {multiplicity.entropy(N_ENTROPY, 0):.3e} J/K   (Omega = 1, fully ordered)")
print(f"S(N, N/2)/kB = {multiplicity.log_multiplicity(N_ENTROPY, N_ENTROPY // 2):.4f}")

## Part 3 — Stirling's approximation and extensivity

$\ln N!$ for macroscopic $N$ is never computed directly; everything above rests on Stirling's
approximation. Measure its error against the exact value, and measure how far entropy is from
being exactly extensive at finite $N$.

In [ ]:
print("Stirling's approximation to ln(N!):")
for n in (10, 100, 1_000, 10_000):
    exact = float(gammaln(n + 1))
    order0 = multiplicity.stirling_log_factorial(n, order=0)
    order1 = multiplicity.stirling_log_factorial(n, order=1)
    print(
        f"  n={n:>6}   exact={exact:12.4f}   |exact-2term|={exact - order0:8.4f}   "
        f"|exact-3term|={exact - order1:10.6f}   1/(12n)={1 / (12 * n):.6f}"
    )

print("\nExtensivity discrepancy: S(2N, N) vs 2 S(N, N/2)")
for n in (200, 2_000, 20_000, 200_000):
    a = multiplicity.entropy(2 * n, n)
    b = 2.0 * multiplicity.entropy(n, n // 2)
    predicted = (0.5 * np.log(np.pi * n) - np.log(2.0)) / (2.0 * n * np.log(2.0))
    print(f"  N={n:>7}   relative_error={relative_error(a, b):.6e}   predicted={predicted:.6e}")

The two-term Stirling formula's *absolute* error grows slowly with $N$ (it is missing the
$\tfrac12\ln(2\pi N)$ term); the three-term form's error shrinks as $1/(12N)$, matching the
next term in the asymptotic series. The extensivity discrepancy shrinks the same way: entropy
is extensive only in the limit, never exactly at finite $N$.

## Part 4 — The Gaussian limit

Expanding $\ln\Omega(N,n)$ to second order about $n=N/2$ gives a Gaussian. Compare it with the
exact combinatorial result near the peak, at a size large enough for the central limit theorem
to bite.

In [ ]:
N_GAUSS = 4000
gauss_counts = np.arange(N_GAUSS // 2 - 300, N_GAUSS // 2 + 301)
exact = np.exp(multiplicity.log_multiplicity_array(N_GAUSS, gauss_counts) - N_GAUSS * np.log(2.0))
gaussian = multiplicity.gaussian_multiplicity_fraction(N_GAUSS, gauss_counts)

max_rel_error = np.max(np.abs(exact - gaussian)) / np.max(exact)
print(f"max relative error near the peak (N = {N_GAUSS}): {max_rel_error:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(gauss_counts, exact, label="exact (binomial)", lw=2, color="#2563eb")
plt.plot(gauss_counts, gaussian, "--", label="Gaussian approximation", lw=1.6, color="crimson")
plt.xlabel("n")
plt.ylabel(r"$\Omega(N,n)/2^N$")
plt.title(f"Central limit theorem, N = {N_GAUSS}")
plt.legend()
plt.tight_layout()
plt.show()

## Part 5 — The Ehrenfest urn

Started from every object on one side, watch the occupancy drift to the even split and stay
there — not because it is forbidden to leave, but because so few of the accessible microstates
sit anywhere else.

In [ ]:
N_URN = 200
occupancy = multiplicity.sample_two_box(N_URN, n_steps=6000, rng=rng)

plt.figure(figsize=(7, 3.5))
plt.plot(occupancy, lw=0.8, color="#2563eb")
plt.axhline(N_URN / 2, color="crimson", ls="--", lw=1.2)
plt.xlabel("step")
plt.ylabel("box A occupancy")
plt.title(f"Ehrenfest urn, N = {N_URN}, started fully in box A")
plt.tight_layout()
plt.show()

late = occupancy[len(occupancy) // 2 :]
print(f"late-time mean fraction   = {late.mean() / N_URN:.4f}   (expect close to 0.5)")
print(f"late-time spread fraction = {late.std() / N_URN:.4f}")

In [ ]:
n_realisations = 24
late_fractions = []
for _ in range(n_realisations):
    trajectory = multiplicity.sample_two_box(N_URN, n_steps=4000, rng=rng)
    late_fractions.append(trajectory[len(trajectory) // 2 :].mean() / N_URN)
late_fractions = np.array(late_fractions)

predicted_sigma = 1.0 / (2 * np.sqrt(N_URN))
print(f"measured std of late-time fraction across {n_realisations} runs: {late_fractions.std(ddof=1):.4f}")
print(f"predicted sigma_n/N = 1/(2 sqrt(N))                          : {predicted_sigma:.4f}")

plt.figure(figsize=(5.5, 3.5))
plt.hist(late_fractions, bins=8, color="#2563eb", edgecolor="white")
plt.axvline(0.5, color="crimson", ls="--")
plt.xlabel("late-time occupancy fraction")
plt.ylabel("count")
plt.title("Spread across independent Ehrenfest runs")
plt.tight_layout()
plt.show()

## Part 6 — Falsifying "entropy is disorder"

A crystal built from a single isotope and a crystal built from a random 50/50 mixture of two
isotopes of the *same* element are, on the same lattice, visually and structurally identical —
same density, same structure, same appearance under any ordinary microscope. Compute their
entropies directly rather than trusting how they look.

In [ ]:
N_SITES = 1000
S_pure = multiplicity.entropy(N_SITES, 0)
S_mixed = multiplicity.entropy(N_SITES, N_SITES // 2)

print(f"pure crystal      (n=0)   : Omega = {multiplicity.multiplicity(N_SITES, 0):.0f}   S = {S_pure:.3e} J/K")
print(
    f"50/50 isotope mix (n=N/2) : ln Omega = "
    f"{multiplicity.log_multiplicity(N_SITES, N_SITES // 2):.2f}   S = {S_mixed:.3e} J/K"
)
print(f"\nThe mixed crystal has {S_mixed - S_pure:.3e} J/K more entropy than the pure crystal,")
print("despite looking visually identical to it.")

assert S_pure == 0.0
assert S_mixed > S_pure

Nothing about the mixed crystal looks more "disordered" than the pure one. Its entropy is
larger because entropy counts accessible microstates, and there are astronomically many ways
to distribute two isotopes across a lattice while producing the same visible macrostate every
time.

## Part 7 — Falsifying "every subsystem's entropy must increase"

Two independent two-state subsystems, $L$ and $R$, are part of one larger isolated system.
Because they are independent, $\Omega_{LR} = \Omega_L \, \Omega_R$, so their entropies add
exactly: $S_{LR} = S_L + S_R$. Nothing about that additivity requires each term to move the
same direction.

In [ ]:
N_SUB = 300
ln_omega_L_before = multiplicity.log_multiplicity(N_SUB, 150)  # L starts at its own peak
ln_omega_L_after = multiplicity.log_multiplicity(N_SUB, 145)  # nudged away from its peak
ln_omega_R_before = multiplicity.log_multiplicity(N_SUB, 280)  # R starts far from its peak
ln_omega_R_after = multiplicity.log_multiplicity(N_SUB, 150)  # R relaxes to its own peak

# S = k_B ln Omega, so a change in ln Omega is a change in S measured in units of k_B.
delta_S_L = ln_omega_L_after - ln_omega_L_before
delta_S_R = ln_omega_R_after - ln_omega_R_before
delta_S_total = delta_S_L + delta_S_R  # exact, because L and R are independent

print(f"Delta S_L / k_B     = {delta_S_L:+.4f}   (subsystem L: entropy DECREASES)")
print(f"Delta S_R / k_B     = {delta_S_R:+.4f}   (subsystem R: entropy increases)")
print(f"Delta S_total / k_B = {delta_S_total:+.4f}   (isolated total: entropy increases)")

assert delta_S_L < 0
assert delta_S_total > 0
print("\nA subsystem's entropy fell while the isolated total's entropy rose.")

## Part 8 — Automated checks

A simulation you have not checked is a picture, not evidence. These are the same assertions
that run in the project's test suite.

In [ ]:
from math import comb, isclose

# 1. Exact combinatorics for small systems.
for n in (1, 2, 10, 25):
    for k in range(n + 1):
        assert isclose(multiplicity.multiplicity(n, k), comb(n, k), rel_tol=1e-9)

# 2. The peak sits at the even split.
peak_counts = np.arange(0, 101)
peak_values = multiplicity.log_multiplicity_array(100, peak_counts)
assert int(peak_counts[np.argmax(peak_values)]) == 50

# 3. Stirling's error bound.
for n in (10, 100, 1000):
    exact = float(gammaln(n + 1))
    approx = multiplicity.stirling_log_factorial(n, order=1)
    assert abs(exact - approx) < 1.0 / (12.0 * n) * 1.01

# 4. The Gaussian limit stays under 1% near the peak at N=4000 (measured in Part 4).
assert max_rel_error < 0.01

# 5. The Ehrenfest urn settles near the even split and stays there (measured in Part 5).
assert abs(late.mean() / N_URN - 0.5) < 0.05

# 6. Entropy of a fully ordered macrostate is exactly zero.
assert multiplicity.entropy(500, 0) == 0.0
assert multiplicity.entropy(500, 500) == 0.0

print("all checks passed")

## Part 9 — Explore it yourself

Slide $N$ and watch the probability distribution over macrostates sharpen around the even
split. Find the smallest $N$ for which you would call the even split "essentially certain,"
and say what standard you used to decide.

In [ ]:
import ipywidgets as widgets
from IPython.display import display


def explore(n_objects=100):
    counts = np.arange(n_objects + 1)
    log_omega = multiplicity.log_multiplicity_array(n_objects, counts)
    prob = np.exp(log_omega - n_objects * np.log(2.0))

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(counts, prob, color="#2563eb", width=1.0)
    ax.axvline(n_objects / 2, color="crimson", ls="--")
    ax.set_xlabel("n")
    ax.set_ylabel("P(n)")
    ax.set_title(
        f"N = {n_objects}   relative peak width = {multiplicity.peak_relative_width(n_objects):.4f}"
    )
    plt.tight_layout()
    plt.show()


widgets.interact(
    explore,
    n_objects=widgets.IntSlider(min=4, max=400, step=2, value=100, description="N"),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "08-multiplicity.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    display_quiz(str(quiz_path))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. The isotope-crystal example showed two visually identical macrostates with wildly different
   entropies. State, in your own words, what entropy actually measures if not visual disorder.
3. Explain, without equations, how a subsystem's entropy can fall while the entropy of the
   isolated total system it belongs to rises.

**Your answers:**

1.
2.
3.